# 03 — Hugging Face parity

Load the HF `PekingU/rtdetr_r{18,50}vd` weights into **our** modules with `strict=True` and check that our outputs match HF's numerically (max abs diff < 1e-4).

Backbone (Task 10), encoder (Task 18) and end-to-end (Task 20) done.

Key mapping lives in `src/utils/hf_weights.py` (`backbone_state_dict`, `encoder_state_dict`).

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

%load_ext autoreload
%autoreload 2

import torch
import transformers
from transformers import RTDetrForObjectDetection
from src.backbone import PResNet
from src.encoder.hybrid_encoder import HybridEncoder
from src.utils.hf_weights import HF_CONFIGS, backbone_state_dict, encoder_state_dict

torch.manual_seed(0)
print("torch", torch.__version__, "| transformers", transformers.__version__)

/mnt/space/conda/envs/dlverse/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch 2.13.0+cu126 | transformers 5.15.0.dev0


## 1. Backbone keys: ours vs. HF

HF backbone keys live under `model.backbone.model.`. After stripping that prefix they must equal our `PResNet` keys exactly (no remapping).

In [2]:
hf = {d: RTDetrForObjectDetection.from_pretrained(f"PekingU/rtdetr_r{d}vd").eval() for d in (18, 50)}

for d in (18, 50):
    ref = backbone_state_dict(hf[d].state_dict())
    ours = PResNet(d, freeze_norm=True).state_dict()
    missing, extra = set(ref) - set(ours), set(ours) - set(ref)
    bad_shapes = [k for k in ref if k in ours and ref[k].shape != ours[k].shape]
    print(f"R{d}: {len(ours)} keys | missing {len(missing)} | extra {len(extra)} | shape mismatches {len(bad_shapes)}")
    assert not missing and not extra and not bad_shapes
print("✅ key sets and shapes identical")

Loading weights:   0%|          | 0/526 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 526/526 [00:00<00:00, 13341.98it/s]

Loading weights:   0%|          | 0/764 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 764/764 [00:00<00:00, 11768.77it/s]

R18: 115 keys | missing 0 | extra 0 | shape mismatches 0
R50: 275 keys | missing 0 | extra 0 | shape mismatches 0
✅ key sets and shapes identical


## 2. Strict load + C3–C5 parity

Same random `[1, 3, 640, 640]` input through ours and `hf.model.backbone.model` (returns `.feature_maps` = C3, C4, C5).

In [3]:
x = torch.randn(1, 3, 640, 640)
backbones = {}

for d in (18, 50):
    ours = PResNet(d, freeze_norm=True).eval()
    print(f"R{d}", ours.load_state_dict(backbone_state_dict(hf[d].state_dict()), strict=True))
    with torch.no_grad():
        ref = hf[d].model.backbone.model(x).feature_maps
        out = ours(x)
    for name, a, b in zip(("C3", "C4", "C5"), out, ref, strict=True):
        diff = (a - b).abs().max().item()
        print(f"   {name} {tuple(a.shape)}  max|diff| = {diff:.2e}")
        assert a.shape == b.shape and diff < 1e-4
    backbones[d] = ours
print("✅ backbone matches HF (R18 + R50)")

R18 <All keys matched successfully>


   C3 (1, 128, 80, 80)  max|diff| = 0.00e+00
   C4 (1, 256, 40, 40)  max|diff| = 0.00e+00
   C5 (1, 512, 20, 20)  max|diff| = 0.00e+00
R50 <All keys matched successfully>


   C3 (1, 512, 80, 80)  max|diff| = 0.00e+00
   C4 (1, 1024, 40, 40)  max|diff| = 0.00e+00
   C5 (1, 2048, 20, 20)  max|diff| = 0.00e+00
✅ backbone matches HF (R18 + R50)


## 3. Encoder keys: ours vs. HF

HF splits our `HybridEncoder` across two places in `RTDetrModel`:

| HF prefix | Our key |
|---|---|
| `model.encoder_input_proj.` | `encoder_input_proj.` (strip `model.` only) |
| `model.encoder.` | strip it (aifi, lateral_convs, fpn_blocks, downsample_convs, pan_blocks) |

In [4]:
for d in (18, 50):
    ref = encoder_state_dict(hf[d].state_dict())
    ours = HybridEncoder(**HF_CONFIGS[d]["encoder"]).state_dict()
    missing, extra = set(ref) - set(ours), set(ours) - set(ref)
    bad_shapes = [k for k in ref if k in ours and ref[k].shape != ours[k].shape]
    print(f"R{d}: {len(ours)} keys | missing {len(missing)} | extra {len(extra)} | shape mismatches {len(bad_shapes)}")
    assert not missing and not extra and not bad_shapes
print("✅ key sets and shapes identical")

R18: 274 keys | missing 0 | extra 0 | shape mismatches 0
R50: 250 keys | missing 0 | extra 0 | shape mismatches 0
✅ key sets and shapes identical


## 4. Strict load + P3 / N4 / N5 parity

HF's own backbone makes C3–C5; the same C3–C5 go into HF (`encoder_input_proj` → `encoder`) and into ours. Square 640×640 and non-square 320×416 (catches a wrong position-embedding order).

In [5]:
encoders = {}

for d in (18, 50):
    m = hf[d].model
    ours = HybridEncoder(**HF_CONFIGS[d]["encoder"]).eval()
    print(f"R{d}", ours.load_state_dict(encoder_state_dict(hf[d].state_dict()), strict=True))
    for (h, w) in [(640, 640), (320, 416)]:
        with torch.no_grad():
            feats = m.backbone.model(torch.randn(1, 3, h, w)).feature_maps           # HF C3, C4, C5
            ref = m.encoder(inputs_embeds=[p(f) for p, f in zip(m.encoder_input_proj, feats)]).last_hidden_state
            out = ours(list(feats))
        diffs = []
        for a, b in zip(out, ref, strict=True):
            assert a.shape == b.shape
            diffs.append((a - b).abs().max().item())
        print(f"   {h}x{w}  P3/N4/N5 {[tuple(a.shape[1:]) for a in out]}  max|diff| = " + " / ".join(f"{x:.2e}" for x in diffs))
        assert max(diffs) < 1e-4
    encoders[d] = ours
print("✅ encoder matches HF (R18 + R50)")

R18 <All keys matched successfully>


   640x640  P3/N4/N5 [(256, 80, 80), (256, 40, 40), (256, 20, 20)]  max|diff| = 2.38e-06 / 1.91e-06 / 1.31e-06


   320x416  P3/N4/N5 [(256, 40, 52), (256, 20, 26), (256, 10, 13)]  max|diff| = 2.38e-06 / 2.15e-06 / 8.64e-07
R50 <All keys matched successfully>


   640x640  P3/N4/N5 [(256, 80, 80), (256, 40, 40), (256, 20, 20)]  max|diff| = 3.34e-06 / 2.86e-06 / 3.46e-06


   320x416  P3/N4/N5 [(256, 40, 52), (256, 20, 26), (256, 10, 13)]  max|diff| = 2.62e-06 / 2.86e-06 / 3.99e-06
✅ encoder matches HF (R18 + R50)


## 5. End-to-end: image → our PResNet → our HybridEncoder vs. HF's full forward

Reference is HF's real `RTDetrForObjectDetection` forward (`encoder_last_hidden_state` = [P3, N4, N5]), not a hand-built HF chain. Shapes must match §7 of HANDOFF: 256 ch at strides 8 / 16 / 32.

In [6]:
n = lambda mod: sum(p.numel() for p in mod.parameters())

for d in (18, 50):
    bb, enc = backbones[d], encoders[d]
    print(f"R{d}: params backbone {n(bb):,} + encoder {n(enc):,} = {n(bb) + n(enc):,}")
    for (h, w) in [(640, 640), (320, 416)]:
        x = torch.randn(1, 3, h, w)
        with torch.no_grad():
            ref = hf[d](pixel_values=x).encoder_last_hidden_state
            out = enc(bb(x))
        diffs = []
        for a, b, s in zip(out, ref, (8, 16, 32), strict=True):
            assert a.shape == b.shape == (1, 256, h // s, w // s)
            diffs.append((a - b).abs().max().item())
        print(f"   {h}x{w}  P3/N4/N5 max|diff| = " + " / ".join(f"{x:.2e}" for x in diffs))
        assert max(diffs) < 1e-4
print("✅ image → PResNet → HybridEncoder matches HF end to end (R18 + R50)")

R18: params backbone 11,190,112 + encoder 4,965,120 = 16,155,232


   640x640  P3/N4/N5 max|diff| = 2.98e-06 / 2.15e-06 / 1.10e-06


   320x416  P3/N4/N5 max|diff| = 2.86e-06 / 2.15e-06 / 9.54e-07
R50: params backbone 23,474,016 + encoder 11,950,848 = 35,424,864


   640x640  P3/N4/N5 max|diff| = 2.86e-06 / 3.58e-06 / 3.16e-06


   320x416  P3/N4/N5 max|diff| = 3.10e-06 / 3.34e-06 / 4.29e-06
✅ image → PResNet → HybridEncoder matches HF end to end (R18 + R50)


## 6. Stretch: our features drive HF's decoder

Pass our [P3, N4, N5] as `encoder_outputs`, so HF skips its own backbone + encoder and runs only query selection + decoder. If our features are right, the 300 detections (class logits + boxes) match HF's own run.

Raw logits go through 6 decoder layers before the sigmoid, so they drift a bit more (~1e-5–1e-4); the asserted values are **scores** (sigmoid) and **boxes**.

In [7]:
for d in (18, 50):
    x = torch.randn(1, 3, 640, 640)
    with torch.no_grad():
        ref = hf[d](pixel_values=x)
        mixed = hf[d](pixel_values=x, encoder_outputs=(encoders[d](backbones[d](x)),))
    dl = (mixed.logits - ref.logits).abs().max().item()
    ds = (mixed.logits.sigmoid() - ref.logits.sigmoid()).abs().max().item()   # scores = what post-processing uses
    db = (mixed.pred_boxes - ref.pred_boxes).abs().max().item()
    same_top = torch.equal(mixed.logits.amax(-1).topk(10).indices, ref.logits.amax(-1).topk(10).indices)
    print(f"R{d}: logits {tuple(ref.logits.shape)} max|diff| = {dl:.2e} | scores {ds:.2e} | boxes max|diff| = {db:.2e} | same top-10 queries: {same_top}")
    assert ds < 1e-4 and db < 1e-4 and same_top
print("✅ HF decoder gives the same detections from our features")

R18: logits (1, 300, 80) max|diff| = 8.08e-05 | scores 2.60e-06 | boxes max|diff| = 6.38e-06 | same top-10 queries: True


R50: logits (1, 300, 80) max|diff| = 3.24e-05 | scores 5.81e-07 | boxes max|diff| = 6.97e-06 | same top-10 queries: True
✅ HF decoder gives the same detections from our features
